# Project Overview

This notebook walks through the full pipeline for training the Chess AI:

1. Filter raw PGN games
2. Convert games into training positions
3. Load dataset for training
4. TODO

Later sections will include:
- Model definition
- Training loop
- Evaluation


# Environment Setup

This notebook installs all required dependencies and prepares the environment.

If you are running this in Google Colab or a fresh environment, run the cell below.

In [1]:
# Install dependencies from requirements.txt
%pip install -r requirements.txt
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Required imports

import chess
import chess.pgn
import json
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from tqdm.notebook import tqdm


# Step 1: Filter High-Quality Games

This step filters raw PGN files to keep only high-quality games.

### Filtering Criteria:
- Both players must have ELO ≥ 2000
- Game must end normally
- Valid result (no abandoned games)
- Minimum number of moves

### Output:
Filtered PGN files are saved to: filtered_games/

In [3]:
# Base directory
BASE_DIR = Path.cwd() / "notebook_outputs"
BASE_DIR.mkdir(exist_ok=True)

# Directory which contains raw pgn files
DATA_DIR = Path.cwd() / "data"

# Directory to store the filtered pgn files
OUTPUT_DIR = BASE_DIR / "filtered_games"

# Minimum number of moves required for keeping
MIN_MOVES = 10

# Minimum ELO for one of the players
MIN_ELO = 2000

# Create the new filtered games directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

def is_valid_game(game):

    """
    Verify that a game is valid and should be kept for training.
    Called in filter_file()
    """

    headers = game.headers

    # Must have ELO ratings
    if not headers["WhiteElo"].isdigit() or not headers["BlackElo"].isdigit():
        return False
    
    # Get the ELO of both players
    white_elo = headers.get("WhiteElo", "?")
    black_elo = headers.get("BlackElo", "?")
    
    # ELO filter 
    #if int(white_elo) < MIN_ELO or int(black_elo) < MIN_ELO:
    #    return False

    # Must end normally
    if headers.get("Termination") != "Normal":
        return False

    # Must have standard result
    if headers.get("Result") not in {"1-0", "0-1", "1/2-1/2"}:
        return False

    # Count moves
    move_count = sum(1 for _ in game.mainline_moves())

    if move_count < MIN_MOVES:
        return False

    return True


def filter_file(input_path, output_path):

    """
    Filter out any undesired games from the input .pgn file
    """

    with open(input_path) as pgn, open(output_path, "w") as outfile:

        game_count = 0  # Total games in file
        kept_count = 0  # Total games kept

        pbar = tqdm(desc=f"Processing {input_path.name}", unit="games")

        while True:

            game = chess.pgn.read_game(pgn)

            # If the game doesn't exist, we've reached the end
            if game is None:
                break

            game_count += 1

            pbar.update(1)

            # If the game is valid, write it to the output file
            if is_valid_game(game):
                kept_count += 1
                print(game, file=outfile, end="\n\n")

        print(f"Finished {input_path}")
        print(f"Kept {kept_count}/{game_count} games")


def main():

    # Loop through all .pgn files
    for filename in os.listdir(DATA_DIR):

        if not filename.endswith(".pgn"):
            continue

        # Get the path to the file
        input_path = DATA_DIR / filename

        output_filename = filename.replace(".pgn", "_filtered.pgn")
        output_path = OUTPUT_DIR / output_filename

        # Skip pgn files that have already been filtered
        if output_path.exists():
            print(f"Skipping {filename} (already filtered)")
            continue

        print(f"Filtering {filename}")

        # Filter the file
        filter_file(input_path, output_path)


main()

Skipping lichess_db_standard_rated_2013-01.pgn (already filtered)
Skipping lichess_db_standard_rated_2013-02.pgn (already filtered)


# Step 2: Convert Games to Training Data

This step converts filtered PGN games into individual board positions.

Each position includes:
- FEN string (board state)
- Player ELO
- Move played
- Game result (value target)

### Output Format:
Saved as `.jsonl` files where each line is:
```json
{
  "fen": "...",
  "elo": 1650,
  "move": "e2e4",
  "value": 1
}

Output Directory: processed_positions/

In [4]:
# Base directory
BASE_DIR = Path.cwd() / "notebook_outputs"
BASE_DIR.mkdir(exist_ok=True)

# Direcotry which contains the filtered games
FILTERED_DIR = BASE_DIR / "filtered_games"

# Directory to contain the processed positions
OUTPUT_DIR = BASE_DIR / "processed_positions"

# Create the new processed positions directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

def result_to_value(result):
    if result == "1-0":
        return 1
    elif result == "0-1":
        return -1
    elif result == "1/2-1/2":
        return 0
    else:
        return None

def process_game(game, outfile):

    board = game.board()

    white_elo = int(game.headers["WhiteElo"])
    black_elo = int(game.headers["BlackElo"])

    for move in game.mainline_moves():
        
        result_str = game.headers.get("Result", "*")
        game_value = result_to_value(result_str)

        # ❗ Skip incomplete games
        if game_value is None:
            return

        fen = board.fen()

        elo = white_elo if board.turn else black_elo

        # CRITICAL: value from perspective of side to move
        position_value = game_value
        if board.turn == chess.BLACK:
            position_value = -position_value

        sample = {
            "fen": fen,
            "elo": elo,
            "move": move.uci(),
            "value": position_value
        }

        outfile.write(json.dumps(sample) + "\n")

        board.push(move)


def process_file(input_path, output_path):

    with open(input_path) as pgn, open(output_path, "w") as outfile:

        game_count = 0
        position_count = 0

        pbar = tqdm(desc=f"Processing {input_path.name}", unit="games")

        while True:

            game = chess.pgn.read_game(pgn)

            if game is None:
                break

            process_game(game, outfile)

            game_count += 1

            pbar.update(1)

        print(f"Finished {input_path}")


def main():

    for input_path in FILTERED_DIR.glob("*.pgn"):

        output_filename = input_path.stem + "_positions.jsonl"
        output_path = OUTPUT_DIR / output_filename

        if output_path.exists():
            print(f"Skipping {input_path.name} (already processed)")
            continue

        print(f"Generating positions from {input_path.name}")

        process_file(input_path, output_path)


main()

Skipping lichess_db_standard_rated_2013-01_filtered.pgn (already processed)
Skipping lichess_db_standard_rated_2013-02_filtered.pgn (already processed)


# Step 3: Utility Functions

These helper functions are used for:
- Board encoding
- Move encoding
- ELO processing
- Legal move masking

They are required before creating the dataset.

## Board Encoding

Converts a chess board into a tensor of shape (18, 8, 8).

Channels include:
- Piece positions (12 channels)
- Side to move
- Castling rights
- En passant square

In [5]:
def board_to_tensor(board):

    tensor = np.zeros((18, 8, 8), dtype=np.float32)

    piece_map = board.piece_map()

    # Chess pieces 
    # True for white, False for black
    piece_to_channel = {
        (chess.PAWN, True): 0,
        (chess.KNIGHT, True): 1,
        (chess.BISHOP, True): 2,
        (chess.ROOK, True): 3,
        (chess.QUEEN, True): 4,
        (chess.KING, True): 5,
        (chess.PAWN, False): 6,
        (chess.KNIGHT, False): 7,
        (chess.BISHOP, False): 8,
        (chess.ROOK, False): 9,
        (chess.QUEEN, False): 10,
        (chess.KING, False): 11
    }

    for square, piece in piece_map.items():

        row = 7 - (square // 8)
        col = square % 8

        channel = piece_to_channel[(piece.piece_type, piece.color)]

        tensor[channel][row][col] = 1

    # Side to move
    tensor[12][:][:] = int(board.turn)

    # Castling rights
    tensor[13][:][:] = int(board.has_kingside_castling_rights(chess.WHITE))
    tensor[14][:][:] = int(board.has_queenside_castling_rights(chess.WHITE))
    tensor[15][:][:] = int(board.has_kingside_castling_rights(chess.BLACK))
    tensor[16][:][:] = int(board.has_queenside_castling_rights(chess.BLACK))

    # En passant
    if board.ep_square is not None:
        row = 7 - (board.ep_square // 8)
        col = board.ep_square % 8
        tensor[17][row][col] = 1

    return tensor

## Move Encoding

Encodes chess moves into a classification space of size 4096.

- Each move = from_square × 64 + to_square

In [6]:
# Map UCI move → class index (0–4095)
def uci_to_class(uci: str) -> int:
    from_square = chess.parse_square(uci[:2])
    to_square = chess.parse_square(uci[2:4])
    return from_square * 64 + to_square

# Map class index → UCI move
def class_to_uci(cls: int) -> str:
    from_square = cls // 64
    to_square = cls % 64
    return chess.SQUARE_NAMES[from_square] + chess.SQUARE_NAMES[to_square]

## ELO Bucketing

Converts raw ELO into discrete skill levels (0–4).

In [7]:
def elo_to_bucket(elo: int) -> int:
    """Convert raw ELO to bucket 0–4"""
    if elo < 800:
        return 0
    elif elo < 1200:
        return 1
    elif elo < 1600:
        return 2
    elif elo < 2000:
        return 3
    else:
        return 4

## Legal Move Masking

Generates a mask of legal moves to prevent illegal predictions.

In [8]:
def get_legal_move_mask(board: chess.Board):
    """
    Returns a mask of shape (4096,)
    1 = legal move
    0 = illegal move
    """
    mask = torch.zeros(4096, dtype=torch.float32)

    for move in board.legal_moves:
        move_class = uci_to_class(move.uci())
        mask[move_class] = 1.0

    return mask

# Step 4: Dataset

This defines a PyTorch Dataset for loading chess positions.

### Features Returned:
- Board tensor (18×8×8)
- Move class (0–4095)
- ELO bucket
- Legal move mask
- Value target

To keep runtime manageable, the dataset is capped at 200,000 samples.

In [9]:
class ChessDataset(Dataset):
    """
    PyTorch Dataset for chess positions.
    Expects a JSONL file where each line is:
    {
        "fen": "...",
        "elo": 1650,
        "move": "e2e4"
    }
    """

    def __init__(self, jsonl_files):
        if isinstance(jsonl_files, (str, Path)):
            jsonl_files = [jsonl_files]

        self.data = []

        MAX_SAMPLES = 200000

        for file in jsonl_files:
            with open(file, 'r') as f:
                for i, line in enumerate(tqdm(f, desc=f"Loading {file}", unit="lines")):
                    # Limit the number of samples
                    if i >= MAX_SAMPLES:
                        break
                    self.data.append(json.loads(line))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        fen = sample['fen']
        move_uci = sample['move']
        elo = sample['elo']

        # Convert FEN to (18,8,8) tensor
        board = chess.Board(fen)
        board_tensor = board_to_tensor(board)
        board_tensor = torch.tensor(board_tensor, dtype=torch.float32)

        # Convert move to class index
        move_class = uci_to_class(move_uci)
        move_class = torch.tensor(move_class, dtype=torch.long)

        # Convert ELO to skill bucket
        elo_bucket = elo_to_bucket(elo)
        elo_bucket = torch.tensor(elo_bucket, dtype=torch.long)

        move_mask = get_legal_move_mask(board)  # (4096,)

        value = torch.tensor(sample['value'], dtype=torch.float32)

        return {
            'board': board_tensor,
            'move_class': move_class,
            'elo_bucket': elo_bucket,
            'move_mask': move_mask,
            'value': value,
        }

## Dataset Sanity Check

Load a small dataset sample and verify everything works correctly.

In [10]:
# Example file
data_path = Path("processed_positions")

files = list(data_path.glob("*.jsonl"))

dataset = ChessDataset(files)

print(f"Dataset size: {len(dataset)}")

sample = dataset[0]

for key, value in sample.items():
    print(f"{key}: {type(value)}, shape: {getattr(value, 'shape', None)}")

Loading processed_positions\lichess_db_standard_rated_2013-01_filtered_positions.jsonl: 0lines [00:00, ?lines/…

Loading processed_positions\lichess_db_standard_rated_2013-02_filtered_positions.jsonl: 0lines [00:00, ?lines/…

Dataset size: 400000
board: <class 'torch.Tensor'>, shape: torch.Size([18, 8, 8])
move_class: <class 'torch.Tensor'>, shape: torch.Size([])
elo_bucket: <class 'torch.Tensor'>, shape: torch.Size([])
move_mask: <class 'torch.Tensor'>, shape: torch.Size([4096])
value: <class 'torch.Tensor'>, shape: torch.Size([])


# Step 6: Model Architecture

This section defines the convolutional neural network used to evaluate chess positions.

---

## Architecture Overview

The model consists of:

### Input Representation
- Board tensor: (18, 8, 8)
- Encodes pieces, turn, castling rights, and en passant

### Feature Extraction
- Initial convolution layer
- Stack of residual blocks (ResNet-style)

### Feature Fusion
- Flattened board features
- Player skill embedding (ELO bucket)
- Material evaluation feature

### Outputs
- **Policy Head**: predicts next move (4096 classes)
- **Value Head**: evaluates position strength ∈ [-1, 1]

---

## Key Design Choices

- Residual connections improve gradient flow
- Skill embedding allows style adaptation
- Material normalization stabilizes training

In [11]:
class ResidualBlock(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)

        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x

        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        x = F.relu(x)

        return x
    
class ChessCNN(nn.Module):

    def __init__(self, num_blocks=6):

        super().__init__()

        channels = 64

        # Initial board processing
        self.input_conv = nn.Conv2d(18, channels, 3, padding=1)
        self.input_bn = nn.BatchNorm2d(channels)

        # Residual stack
        self.res_blocks = nn.Sequential(
            *[ResidualBlock(channels) for _ in range(num_blocks)]
        )

        # Skill embedding
        self.skill_embedding = nn.Embedding(5, 16)

        # Final layers
        # Add 1 to include material
        self.fc1 = nn.Linear(channels * 8 * 8 + 16 + 1, 512)
        
        # Policy head (moves)
        self.policy_head = nn.Linear(512, 4096)

        # Value head (material)
        self.value_head = nn.Linear(512, 1)

    def forward(self, board, skill):

        x = F.relu(self.input_bn(self.input_conv(board)))
        x = self.res_blocks(x)
        x = x.view(x.size(0), -1)

        skill_vec = self.skill_embedding(skill)

        # Material feature
        material = self.compute_material(board)

        x = torch.cat([x, skill_vec, material], dim=1)
        x = F.relu(self.fc1(x))

        policy_logits = self.policy_head(x)
        value = torch.tanh(self.value_head(x))  # constrain to [-1, 1]

        return policy_logits, value
    
    def compute_material(self, board):
        """
        Helper function to compute the material value of pieces still on the board
        board: (B, 18, 8, 8)
        returns: (B, 1)
        """

        # Piece values
        values = torch.tensor([1, 3, 3, 5, 9, 0], device=board.device).view(1, 6, 1, 1)

        # White and black piece planes
        white = board[:, 0:6, :, :]
        black = board[:, 6:12, :, :]

        # Count pieces
        white_count = (white * values).sum(dim=(1,2,3))
        black_count = (black * values).sum(dim=(1,2,3))

        material = white_count - black_count

        # Normalize (important for stability)
        material = material / 39.0  # max theoretical material

        return material.unsqueeze(1)  # (B, 1)

# Step 7: Training Pipeline

This section trains the Chess CNN using the processed dataset.

---

## Training Overview

### Inputs:
- Board tensor
- ELO bucket
- Legal move mask

### Outputs:
- Policy logits (move prediction)
- Value prediction (position strength)

---

## Loss Function

The total loss is a combination of:

- **Policy Loss**: Cross-entropy over legal moves
- **Value Loss**: Mean squared error

Final loss:

Loss = Policy Loss + 0.5 × Value Loss

---

## Training Features

- Legal move masking prevents illegal predictions
- Batch training with PyTorch DataLoader
- Tracks:
  - Loss
  - Accuracy

---

## Outputs

After training, the following are saved:

- Trained model (`.pth`)
- Metrics (`.json`)
- Plots:
  - Loss curve
  - Accuracy curve

In [12]:
def plot_loss_curve(train_losses, save_path):
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss Over Time")
    plt.legend()
    plt.savefig(save_path, dpi=300)
    plt.close()

def plot_accuracy_curve(train_accuracies, save_path):
    plt.figure()
    plt.plot(train_accuracies, label="Train Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training Accuracy Over Time")
    plt.legend()
    plt.savefig(save_path, dpi=300)
    plt.close()

def main():
    # Base directory
    BASE_DIR = Path.cwd() / "notebook_outputs"
    BASE_DIR.mkdir(exist_ok=True)
    
    # Directory which stores the processed positions
    DATA_PATH = BASE_DIR / "processed_positions"

    # Directory which contains the trained CNN model
    MODEL_SAVE_PATH = BASE_DIR / "models" / "chess_cnn_final.pth"

    # Directory to save accuracy metrics
    METRICS_PATH = BASE_DIR / "metrics.json"

    FIGURE_PATH = BASE_DIR / "figures"

    # Hyperparameters (ADJUST)
    BATCH_SIZE = 128
    LEARNING_RATE = 0.001
    EPOCHS = 5

    # Set the training device to CPU
    DEVICE = torch.device("cpu")

    # Load the dataset
    print(f"Loading dataset from: {DATA_PATH}")

    # If the dataset doesn't exist, raise an error
    if not DATA_PATH.exists():
        raise FileNotFoundError(f"Dataset not found at {DATA_PATH}")

    # List every JSONL file in the processed positions directory
    jsonl_files = list(DATA_PATH.glob("*.jsonl"))

    # If there are no JSONL files, raise an error
    if not jsonl_files:
        raise FileNotFoundError(f"No JSONL files found in {DATA_PATH}")

    print(f"Found {len(jsonl_files)} dataset files")

    # Extract the training dataset from the process positions
    train_dataset = ChessDataset(jsonl_files)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

    # Initialize the model
    print("Initializing model...")
    model = ChessCNN(num_blocks=6).to(DEVICE)
    print("Done")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # Loss and accurcy lists used for tracking during training
    train_losses = []
    train_accuracies = []

    # Training Loop
    print("Begin training loop")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", unit="batch")

        for batch in pbar:
            board = batch['board'].to(DEVICE)
            skill = batch['elo_bucket'].to(DEVICE)
            move = batch['move_class'].to(DEVICE)
            mask = batch['move_mask'].to(DEVICE)

            optimizer.zero_grad()
            #outputs = model(board, skill)

            # (B, 4096)
            logits, value_pred = model(board, skill)

            # Apply legal moves mask
            masked_logits = logits.masked_fill(mask == 0, -1e9)

            policy_loss = criterion(masked_logits, move)

            # Loss for material value
            value_criterion = nn.MSELoss()

            value_target = batch['value'].to(DEVICE).float().unsqueeze(1)
            value_loss = value_criterion(value_pred, value_target)

            # Fine tune 0.5
            loss = policy_loss + 0.5 * value_loss

            loss.backward()
            optimizer.step()

            # Track loss
            running_loss += loss.item() * board.size(0)

            # Use masked logits for accuracy
            predicted = masked_logits.argmax(dim=1)
            correct += (predicted == move).sum().item()
            total += move.size(0)

            # Update tqdm bar
            current_loss = running_loss / total if total > 0 else 0
            current_acc = correct / total if total > 0 else 0

            pbar.set_postfix({
                "loss": f"{current_loss:.4f}",
                "acc": f"{current_acc:.4f}"
            })
            
        epoch_loss = running_loss / len(train_dataset)
        epoch_acc = correct / total

        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_acc)

        pbar.close()

        print(f"Epoch {epoch}/{EPOCHS}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

    # Save the model
    print("Saving model...")
    MODEL_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"Model saved to: {MODEL_SAVE_PATH}")

    # Save the loss and accuracy metrics
    metrics = {
        "train_loss": train_losses,
        "train_accuracy": train_accuracies
    }

    with open(METRICS_PATH, "w") as f:
        json.dump(metrics, f, indent=4)

    print(f"Metrics saved to: {METRICS_PATH}")

    # Generate the loss and accuracy plots
    plot_loss_curve(train_losses, FIGURE_PATH / "loss_curve.png")
    plot_accuracy_curve(train_accuracies, FIGURE_PATH / "accuracy_curve.png")

    print("Plots saved to figures/")

main()

Loading dataset from: c:\Users\alexa\source\repos\VT-ECE-5424\ChessAI\notebook_outputs\processed_positions
Found 2 dataset files


Loading c:\Users\alexa\source\repos\VT-ECE-5424\ChessAI\notebook_outputs\processed_positions\lichess_db_standa…

Loading c:\Users\alexa\source\repos\VT-ECE-5424\ChessAI\notebook_outputs\processed_positions\lichess_db_standa…

Initializing model...
Done
Begin training loop


Epoch 1/5:   0%|          | 0/3125 [00:00<?, ?batch/s]

Epoch 1/5, Loss: 3.2235, Accuracy: 0.1906


Epoch 2/5:   0%|          | 0/3125 [00:00<?, ?batch/s]

Epoch 2/5, Loss: 3.0712, Accuracy: 0.2133


Epoch 3/5:   0%|          | 0/3125 [00:00<?, ?batch/s]

KeyboardInterrupt: 

Once the model is trained, you can copy the notebook_outputs/models/ directory to the main project directory. From there, call test_model.py and the entire system will work. Enter FEN board encodings (you can generate them from Lichess.org's board editor: https://lichess.org/editor) and the CNN will work alongside the tree search to predict moves.

## Visualizing Results

This section displays the training loss and accuracy curves.

In [ ]:

BASE_DIR = Path.cwd() / "notebook_outputs"
BASE_DIR.mkdir(exist_ok=True)

METRICS_PATH = BASE_DIR / "metrics.json"

with open(METRICS_PATH, "r") as f:
    metrics = json.load(f)

train_losses = metrics["train_loss"]
train_accuracies = metrics["train_accuracy"]

# Plot side-by-side
fig, axes = plt.subplots(1, 2)

axes[0].plot(train_losses)
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")

axes[1].plot(train_accuracies)
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")

plt.tight_layout()
plt.show()